In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

9. In this exercise, we will predict the number of applications received
using the other variables in the College data set.

(a) Split the data set into a training set and a test set.


In [ ]:
import pandas as pd
import statsmodels.api as sm
college = sm.datasets.get_rdataset("College", "ISLR").data
college.head()
print(college.shape)

                             Private  Apps  Accept  Enroll  Top10perc  \
rownames                                                                
Abilene Christian University     Yes  1660    1232     721         23   
Adelphi University               Yes  2186    1924     512         16   
Adrian College                   Yes  1428    1097     336         22   
Agnes Scott College              Yes   417     349     137         60   
Alaska Pacific University        Yes   193     146      55         16   

                              Top25perc  F.Undergrad  P.Undergrad  Outstate  \
rownames                                                                      
Abilene Christian University         52         2885          537      7440   
Adelphi University                   29         2683         1227     12280   
Adrian College                       50         1036           99     11250   
Agnes Scott College                  89          510           63     12960   
Alaska Pacific

In [ ]:
college['Private'] = college['Private'].map({'Yes':1, 'No':0})

y = college["Apps"]
X = college.drop(columns=["Apps"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
college.head()

,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
rownames,,,,,,,,,,,,,,,,,,
Abilene Christian University,1,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
Adelphi University,1,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
Adrian College,1,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
Agnes Scott College,1,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
Alaska Pacific University,1,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


In [ ]:
print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 543
Test size: 234


(b) Fit a linear model using least squares on the training set, and
report the test error obtained.


In [ ]:
lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred_lm = lm.predict(X_test)
lm_test_mse = mean_squared_error(y_test, y_pred_lm)

print("Linear Model Test MSE:", lm_test_mse)

Linear Model Test MSE: 1371707.7467802023


(c) Fit a ridge regression model on the training set, with λ chosen
by cross-validation. Report the test error obtained.


In [ ]:
scaler = StandardScaler()

ridge = RidgeCV(
    alphas=np.logspace(-3, 5, 100),
    cv=10
)

pipe_ridge = make_pipeline(scaler, ridge)
pipe_ridge.fit(X_train, y_train)

y_pred_ridge = pipe_ridge.predict(X_test)
ridge_mse = mean_squared_error(y_test, y_pred_ridge)

print("Ridge Best Lambda:", ridge.alpha_)
print("Ridge Test MSE:", ridge_mse)


Ridge Best Lambda: 9.111627561154895
Ridge Test MSE: 1190006.6481067121


(d) Fit a lasso model on the training set, with λ chosen by crossvalidation. Report the test error obtained, along with the number of non-zero coefficient estimates.


In [ ]:
lasso = LassoCV(cv=10, random_state=123)

pipe_lasso = make_pipeline(scaler, lasso)
pipe_lasso.fit(X_train, y_train)

y_pred_lasso = pipe_lasso.predict(X_test)
lasso_mse = mean_squared_error(y_test, y_pred_lasso)

coef = pd.Series(lasso.coef_, index=X.columns)
nonzero = coef[coef != 0].index.tolist()

print("Lasso Best Lambda:", lasso.alpha_)
print("Lasso Test MSE:", lasso_mse)
print("Number of non-zero coefficients:", len(nonzero))
print("Selected variables:", nonzero)

Lasso Best Lambda: 3.770815591964155
Lasso Test MSE: 1341725.5402778704
Number of non-zero coefficients: 14
Selected variables: ['Private', 'Accept', 'Enroll', 'Top10perc', 'Top25perc', 'F.Undergrad', 'P.Undergrad', 'Outstate', 'Room.Board', 'PhD', 'Terminal', 'S.F.Ratio', 'Expend', 'Grad.Rate']


(e) Fit a PCR model on the training set, with M chosen by crossvalidation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [ ]:
from sklearn.model_selection import KFold

def pcr_cv(X_train, y_train, max_m=17):
    mse_list = []
    kf = KFold(n_splits=10, shuffle=True, random_state=123)

    for m in range(1, max_m+1):
        pca = PCA(n_components=m)
        X_reduced = pca.fit_transform(X_train)

        mse = -cross_val_score(
            LinearRegression(), X_reduced, y_train,
            cv=kf, scoring="neg_mean_squared_error"
        ).mean()

        mse_list.append(mse)

    best_m = np.argmin(mse_list) + 1
    return best_m, mse_list

best_m, mse_list = pcr_cv(X_train, y_train, max_m=X.shape[1])

print("Best M (number of PCs):", best_m)

Best M (number of PCs): 17


In [ ]:
pca = PCA(n_components=best_m)
X_train_pcr = pca.fit_transform(X_train)
X_test_pcr = pca.transform(X_test)

lm_pcr = LinearRegression()
lm_pcr.fit(X_train_pcr, y_train)

y_pred_pcr = lm_pcr.predict(X_test_pcr)
pcr_mse = mean_squared_error(y_test, y_pred_pcr)

print("PCR Test MSE:", pcr_mse)


PCR Test MSE: 1371707.7467802165


- Ridge regression achieves the lowest test MSE, which is expected because the College dataset contains highly correlated predictors.
- Lasso performs slightly worse due to aggressive variable selection.
- PCR provides similar performance to linear regression, indicating that only a few principal components were selected.

Overall:

Lasso performs feature selection automatically.
Ridge shrinks coefficients but keeps all variables.
PCR reduces dimensionality using components rather than original variables.